In [1]:
import torch
import torch.nn as nn
from datasets import load_dataset
from torch.utils.data import DataLoader
from torchtext.data.utils import get_tokenizer
from collections import defaultdict
from torch.nn.utils.rnn import pad_sequence
from collections import Counter

dataset = load_dataset("imdb")

train_data = dataset["train"]
test_data  = dataset["test"]


print(Counter(train_data["label"]))  
print(Counter(test_data["label"])) 


Counter({0: 12500, 1: 12500})
Counter({0: 12500, 1: 12500})


In [2]:
tokenizer= get_tokenizer("basic_english")

vocab= defaultdict(lambda: 0)

counter=1

for text in dataset["train"]["text"]:
    for token in tokenizer(text):
        if token not in vocab:
            vocab[token]= counter
            counter+=1

vocab_size= len(vocab) + 1


def text_pipeline(text, vocab=vocab):
    return [vocab[token] for token in tokenizer(text)]
    

In [10]:
class SentimentClassifier(nn.Module):
    def __init__(self,vocab_size,embedding_dim,hidden_dim,output_dim):
        super().__init__()

        self.embedding= nn.Embedding(vocab_size, embedding_dim)
        self.lstm= nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc= nn.Linear(hidden_dim, output_dim)

    
    def forward(self, text):

        embedding= self.embedding(text)
        output, (h_n,c_n)= self.lstm(embedding)
        last_hidden_state= h_n[-1]
        out= self.fc(last_hidden_state)

        return out 

device= torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model= SentimentClassifier(vocab_size=vocab_size,embedding_dim=100, hidden_dim=256, output_dim=1)
model= model.to(device)


In [19]:
def collate(batch):

    text_list, label_list= [],[]
    max_len= 256

    for item in batch:

        label_list.append(item['label'])
        processed_text= torch.tensor(text_pipeline(item["text"],vocab)[:max_len], dtype=torch.int64)
        text_list.append(processed_text)

    text_list= pad_sequence(text_list, batch_first=True, padding_value=0)
    label_list= torch.tensor(label_list)

    return label_list, text_list


data_loader= DataLoader(train_data, batch_size=64,shuffle=True, collate_fn=collate)
test_loader= DataLoader(test_data,batch_size=64,shuffle=False, collate_fn=collate)

In [13]:
optimizer= torch.optim.AdamW(model.parameters(), lr=0.001)
criterion= nn.BCEWithLogitsLoss()

num_epochs=10

for epoch in range(num_epochs):

    model.train()
    running_loss=0

    for batch_idx, (label,text) in enumerate(data_loader):

        label= label.float().unsqueeze(1).to(device)
        text=  text.to(device)

        output= model(text)
        loss= criterion(output, label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss+=loss.item()

        if batch_idx % 100 ==0:
            print(f"epoch {epoch+1}/{num_epochs} | batch {batch_idx+1}/{len(data_loader)} | loss {loss.item()}")

    average_loss= running_loss/len(data_loader)
    print(f"average loss of epoch {epoch+1}/{num_epochs} | : {average_loss}")

epoch 1/10 | batch 1/391 | loss 0.6954869627952576
epoch 1/10 | batch 101/391 | loss 0.689273476600647
epoch 1/10 | batch 201/391 | loss 0.6900498867034912
epoch 1/10 | batch 301/391 | loss 0.6847378611564636
average loss of epoch 1/10 | : 0.6930641358160912
epoch 2/10 | batch 1/391 | loss 0.6946042776107788
epoch 2/10 | batch 101/391 | loss 0.6895642280578613
epoch 2/10 | batch 201/391 | loss 0.6928806304931641
epoch 2/10 | batch 301/391 | loss 0.6896893382072449
average loss of epoch 2/10 | : 0.6869310610129705
epoch 3/10 | batch 1/391 | loss 0.6609208583831787
epoch 3/10 | batch 101/391 | loss 0.6608853936195374
epoch 3/10 | batch 201/391 | loss 0.6912922859191895
epoch 3/10 | batch 301/391 | loss 0.6827099919319153
average loss of epoch 3/10 | : 0.6781886448640653
epoch 4/10 | batch 1/391 | loss 0.6437755823135376
epoch 4/10 | batch 101/391 | loss 0.6349484920501709
epoch 4/10 | batch 201/391 | loss 0.6110129356384277
epoch 4/10 | batch 301/391 | loss 0.7920564413070679
average los

In [18]:
def predict_sentiment(text):
    
    with torch.no_grad():
        model.eval()

        text_tensor= torch.tensor(text_pipeline(text, vocab), dtype=torch.int64).unsqueeze(0).to(device)
        prediction= model(text_tensor)
        sentiment= 'positive' if prediction>0.5 else 'negative'

        return sentiment, prediction
    

reviews= [
    'The movie was really great',
    'I really loved the movie',
    'It was really bad',
    'it was okay at its best',
    'Its not bad, but I wouldnt watch it again',
    'I expected it to be terrible… and somehow it wasnt',
    'The acting was great, the plot was painful.'
    
]

for review in reviews:
    sentiment,prediction= predict_sentiment(review)

    print(f"review: {review}")
    print(f"sentiment: {sentiment}")


review: The movie was really great
sentiment: positive
review: I really loved the movie
sentiment: positive
review: It was really bad
sentiment: negative
review: it was okay at its best
sentiment: negative
review: Its not bad, but I wouldnt watch it again
sentiment: positive
review: I expected it to be terrible… and somehow it wasnt
sentiment: positive
review: The acting was great, the plot was painful.
sentiment: negative


In [24]:
total=0
correct=0

with torch.no_grad():
   model.eval()

   for label,text in test_loader:

      label= label.to(device).float()
      text=  text.to(device)

      output= model(text)

      probs= torch.sigmoid(output)
      preds= (probs>=0.5).float()

      total += label.size(0)
      correct += (preds.squeeze()==label).sum().item()

      TP += ((preds == 1) & (label == 1)).sum().item()
      TN += ((preds == 0) & (label == 0)).sum().item()
      FP += ((preds == 1) & (label == 0)).sum().item()
      FN += ((preds == 0) & (label == 1)).sum().item()

   conf_matrix = torch.tensor([[TP, FN],
                               [FP, TN]])
   accuracy= 100 * correct/total

print(accuracy)
print(conf_matrix)

83.596
tensor([[694568, 106072],
        [157768, 642232]])
